In [1]:
from deepanc.common.utils import loading_paths_from_MAT, nmse, apply_impulse_response
from deepanc.inference.antinoise_generator import Deep_ANC_controller
import torch
import numpy as np
import math
import soundfile as sf
from construct_dataset_from_freesound.resample import resampler

In [2]:
from pyprojroot import here
PROJECT_ROOT = here()
MODEL_DIR = PROJECT_ROOT / "models"
PATHS_DIR = PROJECT_ROOT / "rirs"
NOISE_DIR = PROJECT_ROOT / "NOISEX92"
FIG_DIR = PROJECT_ROOT / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)
print(MODEL_DIR)
print(PATHS_DIR)
print(NOISE_DIR)


/home/rkobayashi/anc/deepanc
/home/rkobayashi/anc/deepanc/models
/home/rkobayashi/anc/deepanc/rirs
/home/rkobayashi/anc/deepanc/NOISEX92


In [ ]:
fs = 16000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
delay = 1
rir = [f"rt60_{i}s" for i in [0.15, 0.175, 0.2, 0.225, 0.25]][2]
eta = torch.tensor([math.sqrt(0.1), math.sqrt(1), math.sqrt(10), torch.finfo(torch.float32).max], dtype=torch.float32)[3]
model_name = f"freesound_delay_{delay}.pth"

In [4]:
Pri_path, Sec_path = torch.tensor(loading_paths_from_MAT(PATHS_DIR, rir, "Primary_path.mat", "Secondary_path.mat", )).unsqueeze(1)

sound_names = [ sound_name.name + "/signal.wav" for sound_name in NOISE_DIR.iterdir() if sound_name.is_dir()]

for sound_index in range(len(sound_names)):
    sound_name = sound_names[sound_index]
    print(sound_name.split("/")[0])
    waveform, sample_rate = sf.read(NOISE_DIR/sound_name, dtype="float32", )
    waveform = waveform
    waveform = torch.from_numpy(resampler(waveform, sample_rate, fs))

    Dis, Ref = apply_impulse_response(waveform, Pri_path), waveform
    Time = np.arange(Dis.shape[1]) / fs

    controller = Deep_ANC_controller(MODEL_DIR/model_name, device, fs, delay, eta, Sec_path)
    Error = controller.noise_cancellation(Dis, Ref)

    print(f"  NMSE: {nmse(Error.detach(), Dis.detach())}")

    # Dis, Error = Dis.tolist()[0], Error.tolist()[0]

m109/signal.wav


/tmp/ipykernel_202592/2315813139.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  Pri_path, Sec_path = torch.tensor(loading_paths_from_MAT(PATHS_DIR, rir, "Primary_path.mat", "Secondary_path.mat", )).unsqueeze(1)
/home/rkobayashi/anc/deepanc/src/deepanc/common/utils.py:28: UserWarning: The length of signal is shorter than the length parameter. Result is being padded with zeros in the tail. Please check your center and hop_length settings. (Triggered internally at ../aten/src/ATen/native/SpectralOps.cpp:1166.)
  wave = torch.istft(stft_complex, N_FFT, HOP_LENGTH, window=window, center=False, length=length)


  NMSE: -4.901928424835205
destroyerengine/signal.wav
  NMSE: -8.779898643493652
hfchannel/signal.wav
  NMSE: -7.949510097503662
buccaneer1/signal.wav
  NMSE: -7.839407444000244
factory2/signal.wav
  NMSE: -4.737865447998047
f16/signal.wav
  NMSE: -5.633133888244629
babble/signal.wav
  NMSE: -8.191036224365234
white/signal.wav
  NMSE: -3.8097920417785645
pink/signal.wav
  NMSE: -4.270224094390869
volvo/signal.wav
  NMSE: -3.8960623741149902
leopard/signal.wav
  NMSE: -11.972227096557617
destroyerops/signal.wav
  NMSE: -4.792884826660156
buccaneer2/signal.wav
  NMSE: -5.665186882019043
machinegun/signal.wav
  NMSE: -8.73521614074707
factory1/signal.wav
  NMSE: -5.624102592468262


In [5]:
# import matplotlib.pyplot as plt
# plt.title('The DeepANC Algorithm')
# plt.plot(Time, Dis, color='blue', label='ANC off')
# plt.plot(Time, Error, color='orange', label='ANC on')
# plt.ylabel('Magnitude')
# plt.xlabel('Time (seconds)')
# plt.legend()
# plt.grid()
# plt.savefig(FIG_DIR/'DeepANC.png', dpi=600, bbox_inches='tight', pad_inches=0)
# plt.show()